# C4 — The ring attractor: bump, path integration, and fragility

Companion to
[`part2-case-studies/C4`](../part2-case-studies/C4-continuous-attractors.md).

The cleanest circuit→algorithm conversion in modern neuroscience: ~50 neurons and their
connectivity replaced by **one number** $\theta(t)$ and **one update rule**
$\dot\theta = \omega$.

We build the *Drosophila* ellipsoid-body ring attractor with offset P-EN feedback, verify
that it path-integrates angular velocity with the predicted gain $\kappa\delta/\tau$, and
then break it — because the honest version of the claim is not "this is a ring attractor"
but "this is a ring attractor to within drift of $O(\eta)$, and here is $\eta$."

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

N, tau, dt, T = 128, 0.05, 0.001, 30.0
theta = 2*np.pi*np.arange(N)/N
W0, W1, I0 = -12.0, 4.0, 1.0     # W1 > 2 -> bump exists; W0 < 0 -> global inhibition
delta, kappa = 2*np.pi/N*4, 0.8  # anatomical offset (4 wedges), velocity gain
phi = lambda u: np.maximum(u, 0.0)

def simulate(eta=0.0, seed=0, T=T):
    """Return (t, true angle, decoded bump angle, final bump profile)."""
    rng = np.random.default_rng(seed)
    het = eta*rng.standard_normal((N, N))            # quenched connectivity disorder
    def kernel(shift):
        D = theta[:, None] - theta[None, :] - shift
        return (W0 + W1*np.cos(D))/N * (1.0 + het)
    Kp, Km = kernel(+delta), kernel(-delta)
    t = np.arange(0, T, dt)
    omega = 0.6*np.sin(2*np.pi*t/10.0)               # angular velocity (rad/s)
    u = np.exp(3.0*np.cos(theta)); u /= u.max()
    psi_b = np.zeros_like(t); psi_t = np.zeros_like(t); ang = 0.0
    for n in range(len(t)):
        g = kappa*omega[n]
        W = 0.5*(1+g)*Kp + 0.5*(1-g)*Km
        u = u + (dt/tau)*(-u + W @ phi(u) + I0)
        ang += omega[n]*dt
        psi_t[n] = ang
        psi_b[n] = np.angle(phi(u) @ np.exp(1j*theta))
    return t, psi_t, np.unwrap(psi_b), phi(u)

## 1. The intact ring integrates

The theory says the bump's angular velocity is $\kappa\,\delta/\tau$ times the commanded
velocity. With $\kappa=0.8$, $\delta = 4\cdot 2\pi/128$, $\tau = 0.05$ that is $\pi$.

In [ ]:
t, psi_t, psi_b, bump = simulate(eta=0.0)
gain = np.polyfit(psi_t, psi_b, 1)[0]
print(f'predicted gain kappa*delta/tau = {kappa*delta/tau:.4f}')
print(f'measured gain                  = {gain:.4f}')
print(f'bump width (deg)               = {np.degrees(np.sum(bump > 0)*2*np.pi/N/2):.1f}')
resid = psi_b - psi_t
print(f'residual drift over {T:.0f} s (deg) = {np.degrees(resid[-1]-resid[0]):.3f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(theta, bump, lw=2)
axes[0].set_xlabel('preferred angle (rad)'); axes[0].set_ylabel('activity')
axes[0].set_title('The bump'); axes[0].grid(alpha=.3)
axes[1].plot(t, psi_t, lw=2, label='commanded angle $\\int\\omega\\,dt$')
axes[1].plot(t, psi_b/gain, lw=1.5, ls='--', label='decoded bump / gain')
axes[1].set_xlabel('time (s)'); axes[1].set_ylabel('angle (rad)')
axes[1].set_title('Angular path integration'); axes[1].legend(); axes[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

## 2. Fragility: the continuous attractor is a lie

The continuous ring of fixed points exists only because the connectivity is exactly
rotation-invariant. Any quenched disorder $\eta$ breaks the continuous symmetry, replacing
the ring with a finite set of discrete fixed points and producing systematic drift
$\dot\theta = \omega + g(\theta)$ with $\|g\| = O(\eta)$.

This is the *quantitative* content of the level-2 claim — see
[Orientation Ex 0.4](../00-orientation/README.md).

In [ ]:
etas = [0.0, 0.01, 0.02, 0.05, 0.10]
results = {}
for eta in etas:
    tt, pt, pb, _ = simulate(eta=eta, seed=1)
    r = pb - pt*np.polyfit(pt, pb, 1)[0]/1.0
    drift = np.degrees((pb/np.polyfit(pt, pb, 1)[0] - pt))
    results[eta] = (tt, drift)
    print(f'  eta={eta:5.2f}   peak |drift| = {np.abs(drift-drift[0]).max():7.2f} deg')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for eta, (tt, drift) in results.items():
    ax.plot(tt, drift - drift[0], lw=1.8, label=f'$\\eta$ = {eta}')
ax.set_xlabel('time (s)'); ax.set_ylabel('integration error (deg)')
ax.set_title('Heterogeneity turns an integrator into a leaky, pinned one')
ax.legend(); ax.grid(alpha=.3); plt.tight_layout(); plt.show()

## 3. Where the bump gets pinned

With disorder and *no* velocity input, the bump should relax to one of a small number of
discrete attracting angles rather than staying wherever you put it. That is the direct
signature of a broken continuous attractor.

In [ ]:
def relax(eta, start, seed=1, T=8.0):
    rng = np.random.default_rng(seed)
    het = eta*rng.standard_normal((N, N))
    D = theta[:, None] - theta[None, :]
    W = (W0 + W1*np.cos(D))/N * (1.0 + het)
    u = np.exp(3.0*np.cos(theta - start)); u /= u.max()
    for _ in range(int(T/dt)):
        u = u + (dt/tau)*(-u + W @ phi(u) + I0)
    return np.angle(phi(u) @ np.exp(1j*theta)) % (2*np.pi)

starts = np.linspace(0, 2*np.pi, 40, endpoint=False)
fig, ax = plt.subplots(figsize=(6, 4.2))
for eta, c in zip([0.0, 0.05, 0.15], ['C0', 'C1', 'C3']):
    ends = np.array([relax(eta, s) for s in starts])
    ax.plot(starts, ends, 'o-', ms=4, color=c, label=f'$\\eta$ = {eta}')
ax.plot([0, 2*np.pi], [0, 2*np.pi], 'k:', lw=1, label='no drift (perfect memory)')
ax.set_xlabel('initial bump angle'); ax.set_ylabel('angle after 8 s')
ax.set_title('Staircases = discrete attractors = broken ring')
ax.legend(fontsize=8); ax.grid(alpha=.3); plt.tight_layout(); plt.show()

At $\eta=0$ the curve is the diagonal: every angle is a fixed point, memory is perfect,
the ring attractor is real. As $\eta$ grows the curve develops **steps** — the bump slides
to the nearest surviving discrete attractor. Counting the steps counts the fixed points
that survived, which by Poincaré–Hopf on $S^1$ must come in stable/unstable pairs.

## Things to try

1. **Find the tolerance.** Locate the largest $\eta$ for which drift stays under, say,
   5°/minute — the behaviourally relevant bound. Compare with the ~2% fine-tuning budget
   quoted in C4 §2 for the fly.
2. **Amplitude as a second latent.** Track bump *amplitude* alongside angle. If it is also
   slow, the honest reduction is 2-D, not 1-D — the point of
   [Orientation Ex 0.1](../00-orientation/README.md).
3. **The line-attractor version.** Replace the ring with Seung's oculomotor integrator and
   reproduce the 0.5% tuning problem. Line attractors are strictly harder: a ring at least
   has compactness on its side.
4. **Certify the topology.** Collect bump states across a long run and run persistent
   homology ([S-05](../structures/S-05-toroidal-topology-of-grid-cells.md)). You should
   recover $\beta_0=\beta_1=1$ — the circle — from unlabelled activity alone, which is
   exactly what Chaudhuri et al. did to real head-direction data.